# Aula 16 — Paralelização: fan-out/fan-in de verdade

**Objetivo:** executar ramos em paralelo com correção: quando o join espera,
quando dispara 2x, e como ordenar a mesclagem. Fecha com RAG paralelo real.

**Pré-requisitos:** Aula 02 (arestas) e Aula 08 (reducers).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` a síntese do RAG é real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: cozinha e bar ao mesmo tempo

O pedido sai e **cozinha** e **bar** produzem juntos; a **entrega** junta os
dois quando prontos. Três perguntas que esta aula responde:

| Lanchonete | No LangGraph | Resposta verificada |
|---|---|---|
| cozinha + bar juntos, entrega depois | **fan-out → fan-in** | ramos na mesma etapa rodam juntos |
| entrega espera os dois? | **join simétrico** | sim: 1 execução com tudo |
| bar rápido, cozinha lenta? | **join desigual** | não espera: dispara 1x POR CHEGADA |
| ordem da bandeja | **`sorting_reducer`** | mescla e ordena: determinístico |

A regra de bolso: fan-in espera **dentro da mesma etapa**; chegadas em etapas
diferentes disparam o join de novo.

In [ ]:
# CÉLULA 2 — fan-out simétrico: o join espera e roda 1x
from operator import add
from typing import Annotated, TypedDict
from langgraph.graph import START, END, StateGraph

class Estado(TypedDict):
    itens: Annotated[list, add]

vezes = {'d': 0}

def entrega(state: Estado) -> dict:
    vezes['d'] += 1
    return {'itens': ['D']}

b = StateGraph(Estado)
b.add_node('a', lambda s: {'itens': ['A']})
b.add_node('b', lambda s: {'itens': ['B']})
b.add_node('c', lambda s: {'itens': ['C']})
b.add_node('d', entrega)
b.add_edge(START, 'a')
b.add_edge('a', 'b'); b.add_edge('a', 'c')
b.add_edge('b', 'd'); b.add_edge('c', 'd')
b.add_edge('d', END)
out = b.compile().invoke({'itens': []})
print(out['itens'], '| entrega rodou', vezes['d'], 'x')
assert vezes['d'] == 1 and sorted(out['itens']) == ['A', 'B', 'C', 'D']
print('OK — ramos gêmeos: 1 espera, 1 execução.')

## 2. Conceito: etapa sincroniza, chegada dispara

- O grafo avança em **etapas**: todos os nós prontos rodam 'juntos'.
- Se `b` e `c` terminam **na mesma etapa**, o join `d` roda **1x** com tudo.
- Se `c` termina na etapa 2 e `b2` na etapa 3 (caminhos desiguais), o `d`
  dispara **1x por chegada**: primeiro com o parcial, depois com o completo.
- Consequência prática: join de ramos desiguais precisa ser **idempotente**
  (reexecutável) ou acumulador — nunca 'finalize e feche a conta' na 1ª passada.

In [ ]:
# CÉLULA 3 — fan-out desigual: o join dispara 2x (rode e conte)
vezes2 = {'d': 0}

def entrega2(state: Estado) -> dict:
    vezes2['d'] += 1
    return {'itens': [f"D{vezes2['d']}"]}

b2 = StateGraph(Estado)
b2.add_node('a', lambda s: {'itens': ['A']})
b2.add_node('b', lambda s: {'itens': ['B']})
b2.add_node('b2', lambda s: {'itens': ['B2']})
b2.add_node('c', lambda s: {'itens': ['C']})
b2.add_node('d', entrega2)
b2.add_edge(START, 'a')
b2.add_edge('a', 'b'); b2.add_edge('a', 'c')
b2.add_edge('b', 'b2')
b2.add_edge('b2', 'd'); b2.add_edge('c', 'd')
b2.add_edge('d', END)
out2 = b2.compile().invoke({'itens': []})
print(out2['itens'], '| entrega rodou', vezes2['d'], 'x')
assert vezes2['d'] == 2
print()
print('Leitura: D1 viu o parcial (sem B2); D2 viu tudo. O curso dizia que')
print('o grafo aguardaria: verificado aqui que cada chegada dispara.')

## 3. Dissecando as 2 execuções do join

- 1ª execução: chega `c` (etapa 2). Estado: `[A, B, C]`. `d` roda e anexa `D1`.
- 2ª execução: chega `b2` (etapa 3). Estado: `[A, B, C, B2, D1]`. `d` roda de
  novo e anexa `D2`.
- Repare que a 2ª execução **vê o efeito da 1ª** (`D1` no estado): join
  desigual acumula sobre si mesmo. Se `d` fosse 'fecha a conta e cobra', o
  cliente pagaria 2x — por isso join desigual pede função acumuladora.

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. `sorting_reducer`: ordem determinística, sempre

Com `operator.add`, a ordem dos paralelos varia entre execuções. O reducer
custom que **ordena** elimina a loteria: 3 execuções, 3 resultados iguais.

In [ ]:
# CÉLULA 4 — sorting_reducer (rode 3x e confira)
def sorting_reducer(esquerda, direita):
    if not isinstance(esquerda, list):
        esquerda = [esquerda]
    if not isinstance(direita, list):
        direita = [direita]
    return sorted(esquerda + direita)

class Ordenado(TypedDict):
    itens: Annotated[list, sorting_reducer]

bo = StateGraph(Ordenado)
bo.add_node('x', lambda s: {'itens': ['banana']})
bo.add_node('y', lambda s: {'itens': ['abacaxi']})
bo.add_node('z', lambda s: {'itens': ['caqui']})
bo.add_edge(START, 'x'); bo.add_edge(START, 'y'); bo.add_edge(START, 'z')
bo.add_edge('x', END); bo.add_edge('y', END); bo.add_edge('z', END)
go = bo.compile()
for i in range(3):
    r = go.invoke({'itens': []})['itens']
    print(f'#{i + 1}:', r)
    assert r == ['abacaxi', 'banana', 'caqui'], r
print('OK — determinístico nas 3.')

### 4b. RAG paralelo: 2 fontes juntas + 1 síntese

O padrão real do curso (Wikipedia + Tavily), sem depender de chaves externas:
2 nós-fonte escrevem `contexto` em paralelo, 1 nó sintetiza com o LLM.
Com Groq real a resposta cita os valores; no Fake, conta as fontes.

In [ ]:
# CÉLULA 5 — RAG paralelo (Groq real com chave, Fake sem)
class RAG(TypedDict):
    pergunta: str
    contexto: Annotated[list, add]
    resposta: str

def fonte_cardapio(state: RAG) -> dict:
    return {'contexto': ['cardápio: pastel R$ 8, refri R$ 6']}

def fonte_promocao(state: RAG) -> dict:
    return {'contexto': ['promoção: 2 pastéis por R$ 14']}

def sintetiza(state: RAG) -> dict:
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            pass
        else:
            llm = ChatGroq(model='openai/gpt-oss-20b', temperature=0,
                           reasoning_effort='low')
            r = llm.invoke(f"Com este contexto: {state['contexto']}. Responda: {state['pergunta']}")
            return {'resposta': r.content}
    return {'resposta': f"síntese (Fake) de {len(state['contexto'])} fontes"}

br = StateGraph(RAG)
br.add_node('cardapio', fonte_cardapio)
br.add_node('promocao', fonte_promocao)
br.add_node('sintetiza', sintetiza)
br.add_edge(START, 'cardapio'); br.add_edge(START, 'promocao')
br.add_edge('cardapio', 'sintetiza'); br.add_edge('promocao', 'sintetiza')
br.add_edge('sintetiza', END)
outr = br.compile().invoke({'pergunta': 'quanto custa 2 pastéis?', 'contexto': [], 'resposta': ''})
print('fontes:', sorted(outr['contexto']))
print('resposta:', repr(outr['resposta']))
assert sorted(outr['contexto']) == ['cardápio: pastel R$ 8, refri R$ 6', 'promoção: 2 pastéis por R$ 14']
if os.getenv('GROQ_API_KEY'):
    assert '14' in outr['resposta'], outr['resposta']
    print('OK — síntese real citou a promoção.')
else:
    print('OK — no Fake, as 2 fontes chegaram; a síntese real exige a chave.')

## 5. Erros clássicos

1. **Achar que fan-in sempre espera.** Só dentro da mesma etapa. Ramos
   desiguais re-disparam o join (CÉLULA 3) — e o join vê o próprio efeito anterior.
2. **Join com efeito colateral não-idempotente.** 'Fecha a conta' no join
   desigual cobra 2x. Join acumula; efeito final vai para o END.
3. **Fan-out sem reducer.** Dois escritores na mesma etapa sem `Annotated` =
   `InvalidUpdateError` (Aula 08) — paralelo e reducer andam juntos.
4. **Ordem dos paralelos como garantia.** Com `add` puro, a ordem varia.
   Precisa de ordem? `sorting_reducer` (CÉLULA 4).
5. **Fonte lenta trava a síntese no join simétrico.** O join espera o mais
   lento — timeout por fonte evita 1 fonte trancar tudo.
6. **Ramos que se enxergam.** Paralelos leem o MESMO retrato (Aula 08, seção 3).
   Dependência entre ramos = sequência, não paralelo.

## 6. Exercício guiado — 3 balcões, 1 entrega ordenada

Situação: `padaria`, `acougue` e `adega` produzem em paralelo a partir do
`START`; a `entrega` junta tudo **ordenado** com `sorting_reducer`.
Valide: 1 execução da entrega + lista sempre `['acougue', 'adega', 'padaria']`.

In [ ]:
# CÉLULA 6 — exercício guiado (3 ramos + join ordenado)
class Mercado(TypedDict):
    itens: Annotated[list, sorting_reducer]

vezes_e = {'entrega': 0}

def entrega(state: Mercado) -> dict:
    vezes_e['entrega'] += 1
    return {'itens': ['entrega']}

bm = StateGraph(Mercado)
bm.add_node('padaria', lambda s: {'itens': ['padaria']})
bm.add_node('acougue', lambda s: {'itens': ['acougue']})
bm.add_node('adega', lambda s: {'itens': ['adega']})
bm.add_node('entrega', entrega)
for ramo in ('padaria', 'acougue', 'adega'):
    bm.add_edge(START, ramo)
    bm.add_edge(ramo, 'entrega')
bm.add_edge('entrega', END)
outm = bm.compile().invoke({'itens': []})
print(outm['itens'], '| entrega rodou', vezes_e['entrega'], 'x')
assert vezes_e['entrega'] == 1
assert outm['itens'] == ['acougue', 'adega', 'entrega', 'padaria']
print('OK — 3 ramos juntos, 1 entrega ordenada.')

## 7. Desafio — join desigual consciente

Monte `coleta → (rapido | lento → extra) → junta`, onde `junta` **conta**
quantas vezes rodou e **só finaliza** (escreve `'final'`) quando já viu o item
do ramo longo:

- Dica: `junta` lê o estado (`'extra' in itens?`) e decide: se o longo ainda
  não chegou, anexa `'aguardando'`; se chegou, anexa `'final'`.
- Valide que `junta` rodou 2x e que `'final'` aparece exatamente 1x, por último.
- **Pegadinha:** esse 'join com memória' é um semáforo manual — funciona, mas
  o padrão limpo para N tarefas dinâmicas é `Send` (Aula 19).

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Fan-out simétrico vs desigual: quantas vezes o join roda em cada?
- [ ] O que o join desigual enxerga na 2ª execução (e por que isso morde)?
- [ ] Quando `operator.add` basta e quando precisa de `sorting_reducer`?
- [ ] No RAG paralelo, o que roda junto, o que espera, e quem sintetiza?
- [ ] Join com efeito colateral: por que só acumular, nunca finalizar?
- [ ] Na sua lanchonete, o que paralelizaria (cozinha/bar) e o que juntaria?

**Próxima aula (17 — Sub-grafos):** grafos dentro de grafos — pai que chama
filhos compilados, com `Input/OutputState` para isolar interfaces.